# Módulo 8: Deploy Integral e Testes de Integração de Arquitetura

**Concluído!** A implementação corporativa e governada da **Aria** executando nativamente em toda infraestrutura da AWS AgentCore foi completada. O laboratório 8 finaliza com testes e roteiros seguros para os testes da arquitetura, antes do desmonte infraestrutural de sua Cloud.

## Arquitetura Sistêmica Integral (Review Lógico)

Esta topologia modela integralmente o sistema distribuído provido pela AWS (Módulo 2 ao Módulo 8).

![Overview](../shared/img/08.drawio.png)

### O Cluster do AgentCore AWS Interconectado com o Endpoint React Web

| Ordem Sequencial AWS | Microsserviço Subjacente | Implementação | Propósito de Engenharia |
|---|---|---|---|
| 1 | **Runtime** | Mod 02 | Contêiner Seguro gerenciado (Strands Agent/Python) acoplado a ciclo de vida auto-scale |
| 2 | **Code Interpreter** | Mod 03 | Sandbox Python Isolado - Roda algoritmos, matemática, parseio via bibliotecas nativas |
| 3 | **Browser Tool** | Mod 03 | Navegação remota em Headless Chrome encapsulado via Bedrock Tool. |
| 4 | **Memory** | Mod 04 | Ingestão autônoma em Backend (DynamoDB wrapper). LTM & STM em sincronia restrita ao ID. |
| 5 | **Gateway** | Mod 05 | Acoplamento de Target API via interface MCP gerenciada nativa |
| 6 | **Identity** | Mod 05 | Validador rigoroso do Bearer Token e propulsor da infraestrutura CUSTOM_JWT em nuvem. |
| 7 | **Policy** | Mod 06 | Motor Determinístico base (Cedar) - Interrompe acesso em milissegundos bloqueando a Payload de rede |
| 8 | **Observability** | Mod 07 | Exportador central ADOT - Processo daemon acoplado nativamente na nuvem emitindo AWS X-Ray. |
| 9 | **Evaluations** | Mod 07 | Juízes baseados em IA - Score numérico dinâmico extraído on-the-fly sobre OTel Dumps. |
| -- | **Frontend SPA** | Mod 08 | PWA/WebApp integrando SSE HTTP e OAuth Flow do Cognito. |

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import boto3, json, uuid
from shared import utils

region = utils.get_region()
control = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)

print(f"Region:  {region}")
print(f"Account: {utils.get_account_id()}")

## Validação Forense das APIs (Boto3 SDK)

Teste lógico de Integridade e Dependências via chamadas não-estruturadas Boto3 REST. Nenhum nó pode retornar Null.

In [ ]:
# Verify Runtime
print("=" * 50)
print("  Runtime")
print("=" * 50)

runtime_config = utils.load_config("runtime")
if runtime_config:
    runtime_id = runtime_config["runtime_id"]
    try:
        rt = control.get_agent_runtime(agentRuntimeId=runtime_id)
        print(f"  Name:   {rt.get('agentRuntimeName', 'N/A')}")
        print(f"  ID:     {runtime_id}")
        print(f"  Status: {rt.get('status', 'UNKNOWN')}")
        print(f"  ARN:    {rt.get('agentRuntimeArn', 'N/A')[:80]}...")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Memory
print("=" * 50)
print("  Memory")
print("=" * 50)

memory_config = utils.load_config("memory")
if memory_config:
    memory_id = memory_config["memory_id"]
    try:
        mem = control.get_memory(memoryId=memory_id)
        mem_data = mem.get("memory", mem)
        print(f"  Name:       {mem_data.get('name', 'N/A')}")
        print(f"  ID:         {memory_id}")
        print(f"  Status:     {mem_data.get('status', 'UNKNOWN')}")
        strategies = mem_data.get('memoryStrategies', [])
        print(f"  Strategies: {len(strategies)}")
        for s in strategies:
            for key, val in s.items():
                if isinstance(val, dict):
                    print(f"    - {val.get('name', key)}")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Gateway
print("=" * 50)
print("  Gateway")
print("=" * 50)

gateway_config = utils.load_config("gateway")
if gateway_config:
    gateway_id = gateway_config["gateway_id"]
    try:
        gw = control.get_gateway(gatewayIdentifier=gateway_id)
        print(f"  Name:     {gw.get('name', 'N/A')}")
        print(f"  ID:       {gateway_id}")
        print(f"  Status:   {gw.get('status', 'UNKNOWN')}")
        print(f"  Protocol: {gw.get('protocolType', 'N/A')}")
        print(f"  Auth:     {gw.get('authorizerType', 'N/A')}")
        policy_cfg = gw.get('policyEngineConfiguration', {})
        if policy_cfg:
            print(f"  Policy:   {policy_cfg.get('mode', 'N/A')} mode")
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Policy Engine
print("=" * 50)
print("  Policy Engine")
print("=" * 50)

policy_config = utils.load_config("policy")
if policy_config:
    engine_id = policy_config["policy_engine_id"]
    try:
        engine = control.get_policy_engine(policyEngineId=engine_id)
        print(f"  ID:     {engine_id}")
        print(f"  Status: {engine.get('status', 'UNKNOWN')}")
        print(f"  Mode:   {policy_config.get('enforcement_mode', 'N/A')}")

        # List policies
        try:
            policies = control.list_policies(policyEngineId=engine_id)
            policy_list = policies.get("policies", [])
            for p in policy_list:
                print(f"    - {p.get('name', 'unnamed')} ({p.get('status', '?')})")
            print(f"  Total:  {len(policy_list)} Cedar policies")
        except Exception:
            pass
    except Exception as e:
        print(f"  Error: {e}")
else:
    print("  Not configured")
print()

In [ ]:
# Verify Evaluations
print("=" * 50)
print("  Evaluations")
print("=" * 50)

evals_config = utils.load_config("evaluations")
if evals_config:
    custom_evals = evals_config.get("custom_evaluators", {})
    builtin_evals = evals_config.get("builtin_evaluators", [])
    print(f"  Custom evaluators:  {len(custom_evals)}")
    for name, eid in custom_evals.items():
        print(f"    - {name}: {eid}")
    print(f"  Built-in evaluators: {len(builtin_evals)}")
    for name in builtin_evals:
        print(f"    - {name}")
else:
    print("  Not configured")
print()

## Build & Deploy (SPA Frontend AWS)

Processo de liberação ao usuário final: Acoplar o Backend completo num Web Client seguro e assíncrono.

Os blocos originais (API Gateway regional, Lambdas em python, CloudFront DNS mapping, S3 Block Store e Banco Operacional DynamoDB) nasceram todos provisionados pelo script basilar de pre-requisitos via CloudFormation. A etapa presente injeta todo seu trabalho do `AgentCore` nessa arquitetura estática inicial:

1. **Concessão OAuth no Runtime** — Habilita comunicação Server-Sent Events diretamente de conexões com claims JWT Cognito corretos.
2. **Injeta roteamento no AWS API Gateway** — Aponta a Rota API Rest `/chat` ao Runtime.
3. **Persiste o LTM via Lambda** — O script de memória atualiza a base da lambda do histórico com ID local.
4. **Upload S3 de Build Produtivo Web** — Script escreve config.js em tempo real com ARNs globais e dispara PUT pro Bucket WebS3.
5. **Trigger de Cache Invalidation** — Expurga os arquivos defasados no cluster CDN CloudFront globalmente e injeta na beira os arquivos buildados.

In [ ]:
# Deploy the frontend (Runtime OAuth + API Gateway + S3/CloudFront + Lambda)
# This takes 3-5 minutes on first deploy.

import sys; sys.path.insert(0, 'scripts')
from deploy_frontend import deploy

frontend_config = deploy()

## Ponto de Conexão na Produção Final

Infraestutura totalmente operacional e acessível via Amazon CloudFront. Faça autenticação:

- **Usuário ID:** `workshop@example.com`
- **Master Key:** `WorkshopPass123!`

A validação (End to End Testing E2E) deverá explorar todas as fronteiras de integração dos FMs:
1. **Persistência Neural:** "Salve nas preferências e lembre-se, minha linguagem primária é o python" (Testa Memory Node LTM).
2. **Carga em Sandbox Restrito:** "Construa o fibonacci de 20 e me dê os números resultantes numérico." (Testa o Code Interpreter Serverless CPU).
3. **Integração Target-Role IAM:** "Aloque a tarefa ler docs na cloud AWS". (Chama MCP > Intercepta no Cedar Gateway > Bate na Lambda > Grava DynamoDB).
4. **Conexão Real-Time Internet:** "O que tem de novidade no AWS Re:Invent 2024?" (Roteamento nativo usando Chrome).
5. **Session Dropping:** Renovar Chat inteiro, apagar tela. Inquerir: "Ok, qual é de fato a melhor linguagem de cloud para mim?" (Valida cross-session LTM lookup heurístico).

In [ ]:
# Print the frontend URL and login credentials
frontend_config = utils.load_config("frontend")

if frontend_config:
    url = frontend_config.get("cloudfront_url", "")
    print("=" * 60)
    print("  Aria Web Application")
    print("=" * 60)
    print()
    print(f"  URL:      {url}")
    print()
    print(f"  Username: workshop@example.com")
    print(f"  Password: WorkshopPass123!")
    print()
    print(f"  Open the URL above in your browser to start chatting with Aria.")
    print()
else:
    print("Frontend not deployed yet. Run the deployment cell above first.")

## Relatório Técnico Conclusivo e Recapitulação

A engenharia foi implementada inteiramente no stack modular nativo de Cloud do AWS AgentCore:

| Labs Técnicos Operacionais | Implementação Arquitetural | Entregável Real de Produção |
|---|---|---|
| **00** | Automação e Provisionamento Basilar | Infraestrutura global como Código (CloudFormation e SAM IAM). |
| **01** | Linha de Comando (CLI AWS AgentCore) | Ferramentas essenciais para o loop interno (SDLC Local). |
| **02** | Injeção Lógica no Runtime Python | Contêiner Seguro com SDK AWS. Streaming por default. |
| **03** | Ferramentas de Nuvem AWS Zero-Config | Injeção robusta sem a dor da governança ou instalação local. |
| **04** | Memória Vectorial Heurística e Transacional | Múltiplas instâncias gerando STM & LTM orgânicos e isolados. |
| **05** | Gateway Inteligente em IAM Cognito | Arquitetura segura interligando API legado (via MCP). |
| **06** | Escudo de Barreira Operacional Cedar | Proteções imutáveis que previnem ações destrutivas ou fraudes. |
| **07** | Matriz Completa CloudWatch (Log + Traces) e Auditing Automático | Soluções reais escalonáveis corporativas na Bedrock OTel. |
| **08** | Go-live (Infraestrutura de Interface e Rede) | CloudFront edge servindo a camada final segura para usuário via SSE. |

A arquitetura adota profundamente o paradigma de Single Responsibility (responsabilidade única de microsserviço), o que compõe malhas elásticas perfeitas na nuvem. O código fonte no backend Python segue enxuto: Ele se preocupa com IA e não com a infraestrutura e governança AWS. Essas rotinas sensíveis habitam inteiramente na camada de infra AgentCore Platform.

A infra Web acessa via streaming proxy (Gateway AWS Serverless). A chave-pilar na engenharia está no JWT gerado no login, ele dita os limites de todo acesso interno em Runtime-Downstream. E, novamente, sem que o código altere uma vírgula de arquitetura lógica.

## Limpeza da Infraestrutura Cloud AWS

Terminou? Faça o desmanche (Tear down) preventivo via deleção em CLI. O código a seguir encerra recursos do AgentCore com sucesso e sem rastros tarifados.

> **Atenção (Workshop AWS Integrado):** Em ambientes nativos de Studio o tempo expira automaticamente deletando infraestrutura associada de backend pesado (RDS, S3, lambdas base, cognito etc). Você só precisará executar exclusão nas APIs recém injetadas na nuvem. A AWS varrerá o resto no término da janela operacional.

Sendo seu cluster próprio self-hosted da empresa, os recursos persistentes legados nas `stacks` (cfn e infra visual de nuvem) exigirão o Excluir no Painel de Controle oficial.

In [ ]:
# Delete AgentCore resources (Runtime, Memory, Gateway, Policy Engine)
import sys; sys.path.insert(0, '..'); sys.path.insert(0, 'scripts')
from cleanup import cleanup

# Uncomment the next line to delete all AgentCore workshop resources:
# cleanup(auto_confirm=True)

## Passos Futuros

Com esta especialização e implantação complexa, experimente ampliar os limites arquiteturais:

- **Extensão Severa em Gateways Targets**: Ampliar a IA integrando Sistemas CRM Corporativos (Salesforce), Mensageiros REST e API's ERP via MCP standard AWS.
- **Escalar Segurança em Políticas (Policy/Cedar)**: Limitar acessos na empresa inteira através de Role Based Claims extraídas nativamente do JWT AWS Cognito (Isolando a operação por Time - Admins vs Users).
- **Monitoramento IA Absoluto (Custom Evaluators)**: Escalar Avaliadores com base de referências que garantam que sua corporação e Compliance AWS são estritamente observados pelo Agente Aria (Auditorias rigorosas do Output em tom de Resposta).
- **Nós Distribuídos (Multi-Agent Swarm)**: Ativar infraestrutura na qual diversos perfis autônomos acessam o AgentCore Gateway como elos intercomunicáveis (Modelagem Complexa Multi-IA).
- **Maturidade MLOps/DevOps (Prod)**: Automatizar toda essa esteira contínua nativa através do AWS CodePipeline integrado ao trigger de `deploy_agent.deploy()`. O ambiente AWS lida maravilhosamente bem com deploy nativo contínuo via esteira CI/CD.

### Resources

- [Amazon Bedrock AgentCore Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [Strands Agents SDK](https://github.com/strands-agents/sdk-python)
- [Cedar Language Reference](https://docs.cedarpolicy.com/)
- [AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)
- [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)
- [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
- [AgentCore Policy](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy.html)
- [AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)
- [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html)

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("08")

---

**Parabéns pela Engenharia AWS!** Ao finalizar toda a implementação, a Aria é formalmente uma aplicação servless em modo de Produção Contínua com base nativa nos nove principais serviços do ecossistema Amazon AgentCore.